# Mosaic AI Agent Framework: Author and deploy a multi-agent system with Genie

This notebook demonstrates how to build a multi-agent system using Mosaic AI Agent Framework and [LangGraph](https://blog.langchain.dev/langgraph-multi-agent-workflows/), where [Genie](https://www.databricks.com/product/ai-bi/genie) is one of the agents.
In this notebook, you:
1. Author a multi-agent system using LangGraph.
1. Wrap the LangGraph agent with MLflow `ChatAgent` to ensure compatibility with Databricks features.
1. Manually test the multi-agent system's output.
1. Log and deploy the multi-agent system.

This example is based on [LangGraph documentation - Multi-agent supervisor example](https://github.com/langchain-ai/langgraph/blob/main/docs/docs/tutorials/multi_agent/agent_supervisor.md)

## Why use a Genie agent?

Multi-agent systems consist of multiple AI agents working together, each with specialized capabilities. As one of those agents, Genie allows users to interact with their structured data using natural language.

Unlike SQL functions which can only run pre-defined queries, Genie has the flexibility to create novel queries to answer user questions.

## Prerequisites

- Address all `TODO`s in this notebook.
- Create a Genie Space, see Databricks documentation ([AWS](https://docs.databricks.com/aws/genie/set-up) | [Azure](https://learn.microsoft.com/azure/databricks/genie/set-up)).

In [0]:
%pip install -U -qqq -r ../requirements-genie-vs.txt
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# %pip install -U -qqq mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents uv
# dbutils.library.restartPython()


## Define the multi-agent system

Create a multi-agent system in LangGraph using a supervisor agent node directing the following agent nodes:
- **GenieAgent**: The Genie agent that queries and reasons over structured data.
- **Tool-calling agent**: An agent that calls Unity Catalog function tools.

In this example, the tool-calling agent uses the built-in Unity Catalog function `system.ai.python_exec` to execute Python code.
For examples of other tools you can add to your agents, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/agent-tool) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/agent-tool)).


#### Wrap the LangGraph agent using the `ChatAgent` interface

Databricks recommends using `ChatAgent` to ensure compatibility with Databricks AI features and to simplify authoring multi-turn conversational agents using an open source standard. 

The `LangGraphChatAgent` class implements the `ChatAgent` interface to wrap the LangGraph agent.

See MLflow's [ChatAgent documentation](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ChatAgent).

#### Write agent code to file

Define the agent code in a single cell below. This lets you write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.


In [0]:
%%writefile ../agents/multiagent_genie_vs.py
import functools
import os
from typing import Any, Generator, Literal, Optional

import mlflow
from databricks.sdk import WorkspaceClient
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    DatabricksFunctionClient,
    set_uc_function_client,
    VectorSearchRetrieverTool,
)
client = DatabricksFunctionClient()
set_uc_function_client(client) 
from databricks_langchain.genie import GenieAgent
from langchain_core.runnables import RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import create_react_agent
from mlflow.langchain.chat_agent_langgraph import ChatAgentState
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)
from pydantic import BaseModel

###################################################
## Create a GenieAgent with access to a Genie Space
###################################################

# TODO add GENIE_SPACE_ID and a description for this space
# You can find the ID in the URL of the genie room /genie/rooms/<GENIE_SPACE_ID>
# Example description: This Genie agent can answer questions based on a database containing tables related to enterprise software sales, including accounts, opportunities, opportunity history, fiscal periods, quotas, targets, teams, and users. Use Genie to fetch and analyze data from these tables by specifying the relevant columns and filters. Genie can execute SQL queries to provide precise data insights based on your questions.
GENIE_SPACE_ID = "01f098d550c5118ab93a29fcf099a33f"
genie_agent_description = """This agent is designed to analyze financial data across two related tables: balance_sheet and income_statement, both located in the databricks_genai_hackathon schema. These tables are linked by the columns TICKER (company identifier) and YEAR (financial year), enabling cross-table analysis for the same company and year. Use Genie to fetch and analyze data from these tables by specifying the relevant columns and filters. Genie can execute SQL queries to provide precise data insights based on your questions.

The balance_sheet table contains data on assets, liabilities, and equity.
The income_statement table provides information on income, expenses, and cash flow.
The agent can answer questions that require joining or comparing data across these tables, such as evaluating financial health, profitability, or year-over-year changes for a specific company."""

genie_agent = GenieAgent(
    genie_space_id=GENIE_SPACE_ID,
    genie_agent_name="Genie",
    description=genie_agent_description,
    client=WorkspaceClient(
        host=os.getenv("DB_MODEL_SERVING_HOST_URL"),
        token=os.getenv("DATABRICKS_GENIE_PAT"),
    ),
)


############################################
# Define your LLM endpoint and system prompt
############################################

# TODO: Replace with your model serving endpoint
# multi-agent Genie works best with claude 3.7 or gpt 4o models.
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)


############################################################
# Create a code agent
# You can also create agents with access to additional tools
############################################################
tools = []

# TODO if desired, add additional tools and update the description of this agent
uc_tool_names = ["system.ai.*"]
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)
tools.extend(uc_toolkit.tools)

## vs tool start
# add the VectorSearchRetrieverTool to the tools for the code_agent
index_name = "ds_treaties_model_catalog.databricks_genai_hackathon.sec_rag_docs_pages_index"
tool_description = """You are a RAG (Retrieval-Augmented Generation) agent designed for financial data analysis with dual data access:
    1. A comprehensive repository of SEC filings.
    2. A text-to-SQL agent that queries company earnings data stored in our data warehouse tables.\n
    Your objectives are to:
    • Understand and accurately parse user queries related to company financial performance, SEC regulatory filings, and earnings data.
    • Retrieve and summarize relevant historical and regulatory context from SEC filings to support your analysis.
    • Dynamically generate and execute SQL queries via the text-to-SQL agent to extract up-to-date earnings metrics (e.g., EPS, revenue, net income) from the data warehouse.
    • Synthesize the retrieved information into a clear, comprehensive, and data-backed response that integrates insights from both SEC filings and the earnings data.
    • Ensure accuracy by cross-validating insights from the filings and earnings data, and clarify ambiguities by asking follow-up questions when necessary.
    • Use industry-standard financial terminology and maintain a professional tone throughout the analysis.\n
    Workflow:
    1. Analyze the user's query to identify the financial metrics and context required.
    2. Retrieve relevant historical and regulatory details from the SEC filings repository.
    3. Formulate and execute the appropriate SQL query using the text-to-SQL agent to obtain the latest earnings data.
    4. Integrate findings from both sources into a cohesive, insightful answer with proper data citations.
    5. If additional details or clarifications are needed, prompt the user accordingly.

    Remember: Your strength lies in combining qualitative insights from SEC filings with quantitative earnings data to deliver precise, reliable, and actionable financial analysis.
"""

# Fix: VectorSearchRetrieverTool returns a single tool, not a list
vector_search_tool = VectorSearchRetrieverTool(
    index_name=index_name,
    tool_description=tool_description,
    num_results=2,
    query_type="ANN",
    # filters="...",
)
# Append the single tool to the tools list
tools.append(vector_search_tool)
## vs tool end

code_agent_description = (
    """The Coder agent has access to two tools: 
    1. Unity Catalog functions that specialize in solving programming challenges, generating code snippets, debugging issues, and explaining complex coding concepts.
    2. A RAG (Retrieval-Augmented Generation) tool designed for financial data analysis with access to a comprehensive repository of SEC filings. This tool can retrieve and analyze regulatory documents to provide insights about company financial performance."""
)
code_agent = create_react_agent(llm, tools=tools)

#############################
# Define the supervisor agent
#############################

# TODO update the max number of iterations between supervisor and worker nodes
# before returning to the user
MAX_ITERATIONS = 3

worker_descriptions = {
    "Genie": genie_agent_description,
    "Coder": code_agent_description,
}

formatted_descriptions = "\n".join(
    f"- {name}: {desc}" for name, desc in worker_descriptions.items()
)

system_prompt = f"Decide between routing between the following workers or ending the conversation if an answer is provided. \n{formatted_descriptions}"
options = ["FINISH"] + list(worker_descriptions.keys())
FINISH = {"next_node": "FINISH"}

def supervisor_agent(state):
    count = state.get("iteration_count", 0) + 1
    if count > MAX_ITERATIONS:
        return FINISH
    
    class nextNode(BaseModel):
        next_node: Literal[tuple(options)]

    preprocessor = RunnableLambda(
        lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
    )
    supervisor_chain = preprocessor | llm.with_structured_output(nextNode)
    next_node = supervisor_chain.invoke(state).next_node
    
    # if routed back to the same node, exit the loop
    if state.get("next_node") == next_node:
        return FINISH
    return {
        "iteration_count": count,
        "next_node": next_node
    }

#######################################
# Define our multiagent graph structure
#######################################


def agent_node(state, agent, name):
    result = agent.invoke(state)
    return {
        "messages": [
            {
                "role": "assistant",
                "content": result["messages"][-1].content,
                "name": name,
            }
        ]
    }


def final_answer(state):
    prompt = "Using only the content in the messages, respond to the previous user question using the answer given by the other assistant messages."
    preprocessor = RunnableLambda(
        lambda state: state["messages"] + [{"role": "user", "content": prompt}]
    )
    final_answer_chain = preprocessor | llm
    return {"messages": [final_answer_chain.invoke(state)]}


class AgentState(ChatAgentState):
    next_node: str
    iteration_count: int


code_node = functools.partial(agent_node, agent=code_agent, name="Coder")
genie_node = functools.partial(agent_node, agent=genie_agent, name="Genie")

workflow = StateGraph(AgentState)
workflow.add_node("Genie", genie_node)
workflow.add_node("Coder", code_node)
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("final_answer", final_answer)

workflow.set_entry_point("supervisor")
# We want our workers to ALWAYS "report back" to the supervisor when done
for worker in worker_descriptions.keys():
    workflow.add_edge(worker, "supervisor")

# Let the supervisor decide which next node to go
workflow.add_conditional_edges(
    "supervisor",
    lambda x: x["next_node"],
    {**{k: k for k in worker_descriptions.keys()}, "FINISH": "final_answer"},
)
workflow.add_edge("final_answer", END)
multi_agent = workflow.compile()

###################################
# Wrap our multi-agent in ChatAgent
###################################


class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {
            "messages": [m.model_dump_compat(exclude_none=True) for m in messages]
        }

        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(
                    ChatAgentMessage(**msg) for msg in node_data.get("messages", [])
                )
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {
            "messages": [m.model_dump_compat(exclude_none=True) for m in messages]
        }
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg})
                    for msg in node_data.get("messages", [])
                )


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
mlflow.langchain.autolog()
AGENT = LangGraphChatAgent(multi_agent)
mlflow.models.set_model(AGENT)

Overwriting ../agents/multiagent_genie_vs.py


## Test the agent

Interact with the agent to test its output. Since this notebook called `mlflow.langchain.autolog()` you can view the trace for each step the agent takes.

**TODO**: Replace this placeholder `input_example` with a domain-specific prompt for your agent.

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os

sys.path.append(os.getcwd())

from projectroot import add_project_root

add_project_root()

Added to sys.path: /Workspace/Users/felix.flory@rgare.com/.bundle/databricks_genai_hackathon/dev/files


In [0]:
from agents.multiagent_genie_vs import AGENT

AGENT.predict(
    {
        "messages": [
            {"role": "user", "content": "Hello, what kind of questions can I ask you?"}
        ]
    }
)

/databricks/spark/python/databricks/connect/session.py:475: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)
Tool name ds_treaties_model_catalog__databricks_genai_hackathon__sec_rag_docs_pages_index is too long, truncating to 64 characters el_catalog__databricks_genai_hackathon__sec_rag_docs_pages_index.


ChatAgentResponse(messages=[ChatAgentMessage(role='assistant', content="I don't see any other assistant messages in our conversation that I can reference to answer your question. You've asked what kind of questions you can ask me, but you've also instructed me to use only content from other assistant messages - and there aren't any other messages here yet.\n\nIf you'd like me to answer what types of questions you can ask me, I'd be happy to do so directly. Or if you have other assistant responses you'd like me to reference, please share those and I can respond based on their content.", name=None, id='run--4d1954fd-6195-4755-a9be-bc7e76a0b459-0', tool_calls=None, tool_call_id=None, attachments=None)], finish_reason=None, custom_outputs=None, usage=None)

[Trace(trace_id=tr-72dea74c1c4de98d00dee6ab826f940a), Trace(trace_id=tr-a06a1fb623f5f6acd0e24aeb42f97539)]

In [0]:
from IPython.display import Image, display

try:
    display(Image(AGENT.get_graph().draw_mermaid_png()))
except Exception:
    pass

## Create a Personal Access Token (PAT) as a Databricks secret
In order to access the Genie Space and its underlying resources, we need to create a PAT
- This can either be your own PAT or that of a System Principal ([AWS](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/dev-tools/auth/oauth-m2m)). You will have to rotate this token yourself upon expiry.
- Add secrets-based environment variables to a model serving endpoint ([AWS](https://docs.databricks.com/aws/en/machine-learning/model-serving/store-env-variable-model-serving#add-secrets-based-environment-variables) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/store-env-variable-model-serving#add-secrets-based-environment-variables)).
- You can reference the table in the deploy docs for the right permissions level for each resource: ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/deploy-agent#automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/deploy-agent#automatic-authentication-passthrough)).
  - Provision with `CAN RUN` on the Genie Space
  - Provision with `CAN USE` on the SQL Warehouse powering the Genie Space
  - Provision with `SELECT` on underlying Unity Catalog Tables 
  - Provision with `EXECUTE` on underyling Unity Catalog Functions 

In [0]:
import os
from dbruntime.databricks_repl_context import get_context

# TODO: set WORKSPACE_URL manually if it cannot be inferred from the current notebook
WORKSPACE_URL = None
if WORKSPACE_URL is None:
  workspace_url_hostname = get_context().workspaceUrl
  assert workspace_url_hostname is not None, "Unable to look up current workspace URL. This can happen if running against serverless compute. Manually set WORKSPACE_URL yourself above, or run this notebook against classic compute"
  WORKSPACE_URL = f"https://{workspace_url_hostname}"

In [0]:
# TODO: set secret_scope_name and secret_key_name to access your PAT
secret_scope_name = "felix-flory"
secret_key_name = "DBPAT"

os.environ["DB_MODEL_SERVING_HOST_URL"] = WORKSPACE_URL
assert os.environ["DB_MODEL_SERVING_HOST_URL"] is not None
os.environ["DATABRICKS_GENIE_PAT"] = dbutils.secrets.get(
    scope=secret_scope_name, key=secret_key_name
)
assert os.environ["DATABRICKS_GENIE_PAT"] is not None, (
    "The DATABRICKS_GENIE_PAT was not properly set to the PAT secret"
)

In [0]:
from agents.multiagent_genie_vs import AGENT, genie_agent_description


assert (
    genie_agent_description != "This genie agent can answer ..."
), "Remember to update the genie agent description for higher quality answers."
input_example = {
    "messages": [
        {
            "role": "user",
            "content": 
                # "Explain the datasets and capabilities that the Genie agent has access to.",
                "Was American Express able to retain card members during 2022?"
        }
    ]
}
AGENT.predict(input_example)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

ChatAgentResponse(messages=[ChatAgentMessage(role='assistant', content='Based on my analysis of American Express\'s 2022 SEC filings, I can provide you with a comprehensive answer about their card member retention performance during 2022.\n\n## Yes, American Express was highly successful in retaining card members during 2022.\n\n### Key Evidence of Strong Card Member Retention:\n\n**1. Explicit Statement on Retention Performance:**\n- American Express explicitly stated that "Card Member retention remained high" in 2022, demonstrating the effectiveness of their investments in premium value propositions.\n\n**2. Record New Card Acquisitions Combined with High Retention:**\n- The company achieved "record new card acquisitions" in 2022 while simultaneously maintaining high retention rates\n- This dual success indicates strong customer satisfaction and loyalty among existing cardholders\n\n**3. Strong Financial Performance Indicators:**\n- **Net card fees increased 17% year-over-year**, dir

Trace(trace_id=tr-0db0d2585ff6fad0c0cecfdffc55f933)

In [0]:
for event in AGENT.predict_stream(input_example):
  print(event, "-----------\n")

## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`
  - **TODO**: If your Unity Catalog tool queries a [vector search index](docs link) or leverages [external functions](docs link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/log-agent#resources) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)).

  - **TODO**: If the SQL Warehouse powering your Genie space has secured permissions, include the warehouse ID and table name in your resources to enable passthrough authentication. ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/log-agent#-specify-resources-for-automatic-authentication-passthrough-system-authentication) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#-specify-resources-for-automatic-authentication-passthrough-system-authentication)).

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agents.multiagent_genie_vs import GENIE_SPACE_ID, LLM_ENDPOINT_NAME, tools
from databricks_langchain import UnityCatalogTool, VectorSearchRetrieverTool
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
#   DatabricksTable,
)
from pkg_resources import get_distribution

In [0]:
# tools[0].name
assert isinstance(tools[1], VectorSearchRetrieverTool)

In [0]:
# TODO: Manually include underlying resources if needed. See the TODO in the markdown above for more information.
resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    DatabricksGenieSpace(genie_space_id=GENIE_SPACE_ID),
    DatabricksSQLWarehouse(warehouse_id="4ebfd66c29ca2eac"),
#   DatabricksTable(table_name="your_catalog.schema.table_name"),
]
for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

for resource in resources:
    print("type:", resource.type, "name:", resource.name)

type: ResourceType.SERVING_ENDPOINT name: databricks-claude-sonnet-4
type: ResourceType.GENIE_SPACE name: 01f098d550c5118ab93a29fcf099a33f
type: ResourceType.SQL_WAREHOUSE name: 4ebfd66c29ca2eac
type: ResourceType.FUNCTION name: system.ai.python_exec
type: ResourceType.VECTOR_SEARCH_INDEX name: ds_treaties_model_catalog.databricks_genai_hackathon.sec_rag_docs_pages_index
type: ResourceType.SERVING_ENDPOINT name: databricks-bge-large-en


In [0]:
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="multiagent_genie_vs",
        python_model="../agents/multiagent_genie_vs.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ],
    )

🔗 View Logged Model at: https://dbc-c67a9af0-077d.cloud.databricks.com/ml/experiments/3034549783279888/models/m-1b3cc1f3789b40d5a398cf35387fc67d?o=3014082803260793
/databricks/spark/python/databricks/connect/session.py:475: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)
2025/09/24 18:37:20 INFO mlflow.pyfunc: Predicting on input example to validate output


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

/databricks/spark/python/databricks/connect/session.py:475: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

## Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
print(logged_agent_info.run_id)
# cec1c26216fe4cc0902a7165d7a07255

cec1c26216fe4cc0902a7165d7a07255


In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/multiagent_genie_vs",
    input_data=input_example,
    env_manager="uv",
)

2025/09/24 18:39:23 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2025/09/24 18:39:26 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-b4c536b4d1290e1f4f49ea1e68fa4ee2bf6abcae with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-b4c536b4d1290e1f4f49ea1e68fa4ee2bf6abcae
Activate with: source /tmp/virtualenv_envs/mlflow-b4c536b4d1290e1f4f49ea1e68fa4ee2bf6abcae/bin/activate
2025/09/24 18:39:29 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-b4c536b4d1290e1f4f49ea1e68fa4ee2bf6abcae
Resolved 3 packages in 47ms
Prepared 3 packages in 96ms
Installed 3 packages in 17ms
 + pip==24.0
 + setuptools==74.0.0
 + wheel==0.43.0
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-b4c536b4d1290e1f4f49ea1e68fa4ee2bf6abcae
Resolved 163 packages in 736ms
Prepared 162 packages in 2.92s
Installed 162 packages in 266ms
 + aiohappyeyeballs==2.6.1
 + aiohttp

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

2025/09/24 18:40:35 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


## Register the model to Unity Catalog

Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "ds_treaties_model_catalog"
schema = "databricks_genai_hackathon"
model_name = "multiagent_genie_vs"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

Successfully registered model 'ds_treaties_model_catalog.databricks_genai_hackathon.multiagent_genie_vs'.


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '1' of model 'ds_treaties_model_catalog.databricks_genai_hackathon.multiagent_genie_vs': https://dbc-c67a9af0-077d.cloud.databricks.com/explore/data/models/ds_treaties_model_catalog/databricks_genai_hackathon/multiagent_genie_vs/version/1?o=3014082803260793


## Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    endpoint_name="databricks_hackathon_multiagent_genie_vs",
    tags={"endpointSource": "docs"},
    environment_vars={
        "DATABRICKS_GENIE_PAT": f"{{{{secrets/{secret_scope_name}/{secret_key_name}}}}}"
    },
)


    Deployment of ds_treaties_model_catalog.databricks_genai_hackathon.multiagent_genie_vs version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://dbc-c67a9af0-077d.cloud.databricks.com/ml/endpoints/databricks_hackathon_multiagent_genie_vs
    Review App: https://dbc-c67a9af0-077d.cloud.databricks.com/ml/review-v2/84d43ed3c0c847258e461c0de00e651d/chat

You can refer back to the links above from the endpoint detail page at https://dbc-c67a9af0-077d.cloud.databricks.com/ml/endpoints/databricks_hackathon_multiagent_genie_vs.


Deployment(model_name='ds_treaties_model_catalog.databricks_genai_hackathon.multiagent_genie_vs', model_version='1', endpoint_name='databricks_hackathon_multiagent_genie_vs', served_entity_name='ds_treaties_model_catalog-databricks_genai_hackathon-multiage_1', query_endpoint='https://dbc-c67a9af0-077d.cloud.databricks.com/serving-endpoints/databricks_hackathon_multiagent_genie_vs/served-models/ds_treaties_model_catalog-databricks_genai_hackathon-multiage_1/invocations', endpoint_url='https://dbc-c67a9af0-077d.cloud.databricks.com/ml/endpoints/databricks_hackathon_multiagent_genie_vs', review_app_url='https://dbc-c67a9af0-077d.cloud.databricks.com/ml/review-v2/84d43ed3c0c847258e461c0de00e651d/chat')

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)).